# ACTO SDK: Basic Proof Creation

This notebook demonstrates how to create and verify execution proofs using the ACTO SDK.


In [ ]:
from acto.proof import create_proof, verify_proof
from acto.telemetry.models import TelemetryBundle, TelemetryEvent
from acto.crypto import KeyPair
from datetime import datetime


## Generate a Keypair


In [ ]:
# Generate a new Ed25519 keypair
keypair = KeyPair.generate()
print(f"Public key: {keypair.public_key_b64}")


## Create Telemetry Bundle


In [ ]:
# Create sample telemetry data
telemetry_events = [
    TelemetryEvent(
        ts=datetime.now().isoformat(),
        topic="lidar",
        data={"distance": 1.5, "angle": 45.0}
    ),
    TelemetryEvent(
        ts=datetime.now().isoformat(),
        topic="camera",
        data={"image_id": "img_001", "detections": 3}
    )
]

# Create telemetry bundle
bundle = TelemetryBundle(
    task_id="cleaning-run-001",
    robot_id="robot-001",
    run_id="run-001",
    events=telemetry_events
)

print(f"Created bundle for task: {bundle.task_id}")


## Create Proof


In [ ]:
# Create signed proof envelope
envelope = create_proof(
    bundle,
    keypair.private_key_b64,
    keypair.public_key_b64
)

print(f"Proof created!")
print(f"Payload hash: {envelope.payload.payload_hash}")
print(f"Task ID: {envelope.payload.subject.task_id}")


## Verify Proof


In [ ]:
# Verify the proof
try:
    is_valid = verify_proof(envelope)
    print(f"✓ Proof is valid: {is_valid}")
except Exception as e:
    print(f"✗ Proof verification failed: {e}")


## Save Proof to File


In [ ]:
import json
from pathlib import Path

# Save proof to file
output_path = Path("proof.json")
output_path.write_text(envelope.model_dump_json(indent=2))
print(f"Proof saved to: {output_path}")
